# 七状态转向轨迹误差 LQR

In [1]:
from pathlib import Path
import sys
root=Path.cwd()
while not (root/'base.py').exists():
    if root==root.parent: raise RuntimeError('找不到 theory/base.py')
    root=root.parent
sys.path.insert(0,str(root))
import sympy as s
import numpy as np
from IPython.display import display, Math
# 只改 LaTeX 显示，不替换计算符号：dx -> x 上点，ddx -> x 双点。
from base import q, qd, qdd
latex_names = {}
for coordinate, velocity, acceleration in zip(q, qd, qdd):
    name = s.latex(coordinate)
    latex_names[velocity] = r'\dot{' + name + '}'
    latex_names[acceleration] = r'\ddot{' + name + '}'
s.init_printing(symbol_names=latex_names)
np.set_printoptions(precision=8,suppress=True,linewidth=130)
def show(label, expression):
    display(Math(label+' = '+s.latex(expression, symbol_names=latex_names)))


In [2]:
from turning_balance.lqr import *
result=design()

## 世界状态与参考轨迹
$$Z=[x,y,\psi,\theta,v,\omega,p]^T,\quad
\dot Z=[v\cos\psi,v\sin\psi,\omega,p,(M^{-1}(B_\tau\tau-h))^T]^T.$$
参考选择已有实验的 $v_0=0.1\,m/s,\omega_0=0.4\,rad/s$，$\theta_r=p_r=0$，理想无耗散模型的 $\tau_r=0$。
$\psi_r=\psi_0+\omega_0t$，$x_r,y_r$ 为圆周轨迹；小量是**相对轨迹的误差**，不是世界位置或偏航角自身。

$$\delta\dot Z=A_w(t)\delta Z+B_w\delta\tau.$$

下面 A 的 `psi_ref` 随时间改变；不能把一个时刻的 A 当作整个轨迹的常矩阵。

In [3]:
sy=symbolic()
show('A_w(t)',sy['A']);show('B_w',sy['B'])

<IPython.core.display.Math object>

<IPython.core.display.Math object>

## 旋转位置误差得到常系数形式
$$E=H(t)\delta Z=[e_{\parallel},e_{\perp},e_\psi,e_\theta,e_v,e_\omega,e_p]^T.$$
H 只旋转前两项，不旋转角度和速度误差。
$$A_e=H A_w H^{-1}+\dot H H^{-1},\quad B_e=H B_w.$$
对于此处恒定 $v_0,\omega_0$，$A_e,B_e$ 是常矩阵。$\dot H$ 不能省略。
$Q$ 的两个位置权重相同，因此这个选择也等价于世界位置误差的各向同性惩罚。

In [4]:
show('H',sy['H']);show('A_e',sy['Ae']);show('B_e',sy['Be'])

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

## 从性能指标到最终增益
下列 $E$ 表示所选线性误差状态（静止分支就是 $X$，转向分支是旋转坐标中的七维误差）。

$$\dot E=A_eE+B_e\delta\tau,\qquad
J=\int_0^\infty(E^TQE+\delta\tau^TR\delta\tau)\,dt.$$

$Q\succeq0$ 惩罚各状态偏差，$R\succ0$ 惩罚电机力矩。权重有量纲依赖，不是车辆物理参数。
在可稳定、可检测的线性问题上，设最小剩余代价 $V(E)=E^TPE$。

$$0=\min_{\delta\tau}\{E^TQE+\delta\tau^TR\delta\tau+2E^TP(A_eE+B_e\delta\tau)\}.$$

对力矩求导：$2R\delta\tau+2B_e^TPE=0$，故

$$\delta\tau_*=-R^{-1}B_e^TPE=-K_e E.$$

代回并比较二次型系数，得到 CARE：

$$A_e^TP+PA_e-PB_eR^{-1}B_e^TP+Q=0.$$

选择使 $A_e-B_eK_e$ 稳定的对称解 $P$，于是 $J_{\min}(E_0)=E_0^TPE_0$。
这给出最优反馈，而不是把积分式对常矩阵 $K$ 的每个元素蛮力求极值。
实际程序用 `solve_continuous_are` 求 $P$，用 `solve(R, B.T@P)` 得到 $K$。
下方展示 $Q,R,P,K$ 的实际数值及残差。该最优性只针对所写的线性误差系统和无饱和二次代价。

参考：[MIT Underactuated Robotics — LQR](https://underactuated.mit.edu/lqr.html)。


In [5]:
from parameters import values
print('临时示例参数（不是实车值）：',values()[0])
for key in ('A','B','Q','R','P','K'):show(key,s.Matrix(result[key]).evalf(8))
print('可控性秩：',result['rank'])
print('闭环特征值：',result['poles'])
print('CARE残差：',result['care_residual'])

临时示例参数（不是实车值）： {'r': 0.1, 'd': 0.4, 'l': 0.3, 'm_B': 2.0, 'm_L': 0.5, 'm_R': 0.5, 'I_Lyy': 0.0025, 'I_Ryy': 0.0025, 'I_Lzz': 0.00125, 'I_Rzz': 0.00125, 'I_Byy': 0.08, 'I_Bzz': 0.06, 'g': 9.81, 'psi0': 0.0, 'I_Bxx': 0.1, 'I_Lxx': 0.00125, 'I_Rxx': 0.00125}


<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

可控性秩： 7
闭环特征值： [-49.85054557+0.j         -10.20103834+0.j          -3.19930872+0.j          -0.66191194+0.68235488j  -0.66191194-0.68235488j
  -0.46811818+0.j          -1.59492507+0.j        ]
CARE残差： 4.974465284135476e-12


## 最优代价示例
只给初始俯仰误差 0.03 rad，其他误差为0；下面得到该线性无限时域问题的最小代价数值。

In [6]:
E0=np.zeros(len(result['A']));E0[3 if len(E0)==7 else 2]=.03
show('J_{min}',s.Float(E0@result['P']@E0))

<IPython.core.display.Math object>

## 世界坐标反馈与适用边界
$$\tau=\tau_r-K_eE=\tau_r-K_w(t)\delta Z,\quad K_w(t)=K_eH(t).$$
K_e 是上面的 2×7 常矩阵，每一行对应一侧电机、每列对应一种误差。
下面列出参考朝向 0.7 rad 时的世界 A、B、K，验证矩阵不是只支持零偏航。
若 $v_0=\omega_0=0$，横向位置出现不可控零模态，不能用正定位置代价得到稳定全部七状态的 CARE 解；静止平衡请使用六状态分支。
一般加减速参考不满足常系数条件，需重新沿可行轨迹线性化并处理时变 Riccati 问题。
当前 ROS `balance_yaw` 仍使用已验收的六状态里程控制器，本次七状态理论并未自动替换它。

In [7]:
Aw,Bw,Ae,H=state_matrices(heading=.7)
show('A_w(0.7)',s.Matrix(Aw).evalf(8));show('B_w',s.Matrix(Bw).evalf(8))
show('K_w(0.7)',s.Matrix(result['K']@H).evalf(8))
A0,B0,_,_=state_matrices(0,0)
print('静止七状态可控性秩：',np.linalg.matrix_rank(np.hstack([np.linalg.matrix_power(A0,i)@B0 for i in range(7)])))

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

静止七状态可控性秩： 6
